# Latent-noise watermark test (detection only)

Runs images backwards through a **public** Stable Diffusion model (DDIM inversion) and checks the recovered starting noise for watermark structure (Tree-Ring-style rings, non-Gaussian statistics).

**Positive control first:** we make our own Tree-Ring-marked images with our own random key and check the test finds them. If the control fails, ignore the Kling result.

Before running: `Runtime > Change runtime type > T4 GPU`. Then run each cell in order.

In [ ]:
!pip -q install diffusers transformers accelerate scipy
import torch, numpy as np, glob, zipfile, os
from PIL import Image
from scipy.stats import mannwhitneyu, kstest
assert torch.cuda.is_available(), 'Switch the runtime to a GPU (T4) first'
print(torch.cuda.get_device_name(0))

In [ ]:
# Upload samples/colab_upload.zip from your Mac when the button appears
from google.colab import files
up = files.upload()
zipfile.ZipFile(next(iter(up))).extractall('.')
print({d: len(glob.glob(f'colab_upload/{d}/*.png')) for d in ('kling', 'gemini')})

In [ ]:
from diffusers import StableDiffusionPipeline, DDIMScheduler, DDIMInverseScheduler
pipe = None
for mid in ['stable-diffusion-v1-5/stable-diffusion-v1-5', 'sd-legacy/stable-diffusion-v1-5', 'stabilityai/stable-diffusion-2-1-base']:
    try:
        pipe = StableDiffusionPipeline.from_pretrained(mid, torch_dtype=torch.float16, safety_checker=None).to('cuda'); print('model:', mid); break
    except Exception as e:
        print('could not load', mid, type(e).__name__)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
inv = DDIMInverseScheduler.from_config(pipe.scheduler.config)
EMB = pipe.encode_prompt('', 'cuda', 1, False)[0]

@torch.no_grad()
def invert(img, steps=50):
    x = torch.from_numpy(np.asarray(img.convert('RGB').resize((512, 512)))).float() / 127.5 - 1
    x = x.permute(2, 0, 1)[None].to('cuda', torch.float16)
    lat = pipe.vae.encode(x).latent_dist.mean * pipe.vae.config.scaling_factor
    inv.set_timesteps(steps)
    for t in inv.timesteps:
        eps = pipe.unet(lat, t, encoder_hidden_states=EMB).sample
        lat = inv.step(eps, t, lat).prev_sample
    return lat.float().cpu().numpy()[0]

H = W = 64
yy, xx = np.mgrid[:H, :W]; RAD = np.sqrt((yy - H // 2) ** 2 + (xx - W // 2) ** 2)
def ring_score(z, R=10):
    # Tree-Ring keys make each ring of the latent FFT (nearly) constant -> low spread inside rings
    best = []
    for ch in range(4):
        F = np.fft.fftshift(np.fft.fft2(z[ch]))
        vals = [np.var(F[(RAD >= k - 0.5) & (RAD < k + 0.5)].real) / (np.mean(np.abs(F[(RAD >= k - 0.5) & (RAD < k + 0.5)]) ** 2) + 1e-9) for k in range(1, R + 1)]
        best.append(np.mean(vals))
    return float(min(best))   # lower = more ring-like
def gauss_score(z):
    # distance of the recovered noise from a standard Gaussian (larger = less like natural starting noise)
    v = z.ravel(); v = (v - v.mean()) / (v.std() + 1e-9)
    return float(kstest(v, 'norm').statistic)
print('ready')

In [ ]:
# Positive control: our own Tree-Ring-style key (detection test only)
def marked_latents(seed, R=10, ch=3):
    g = torch.Generator('cuda').manual_seed(seed)
    z = torch.randn((1, 4, 64, 64), generator=g, device='cuda')
    F = torch.fft.fftshift(torch.fft.fft2(z), dim=(-1, -2))
    r = torch.from_numpy(RAD).cuda()
    key = torch.randn(R, generator=g, device='cuda') * 64
    for k in range(1, R + 1):
        F[0, ch][(r >= k - 0.5) & (r < k + 0.5)] = key[k - 1].to(F.dtype)
    return torch.fft.ifft2(torch.fft.ifftshift(F, dim=(-1, -2))).real.half()
prompts = ['a forest floor with leaves', 'a busy city street', 'pebbles on a beach', 'an old brick wall', 'a field of flowers', 'a library with books']
ctrl_marked, ctrl_clean = [], []
for i, p in enumerate(prompts):
    ctrl_marked.append(pipe(p, latents=marked_latents(100 + i), num_inference_steps=50, guidance_scale=7.5).images[0])
    ctrl_clean.append(pipe(p, generator=torch.Generator('cuda').manual_seed(200 + i), num_inference_steps=50, guidance_scale=7.5).images[0])
print('made', len(ctrl_marked), 'marked and', len(ctrl_clean), 'clean control images')

In [ ]:
groups = {
    'SD clean (no mark)': ctrl_clean,
    'SD Tree-Ring (POSITIVE CONTROL)': ctrl_marked,
    'Gemini (SynthID, not latent)': [Image.open(f) for f in sorted(glob.glob('colab_upload/gemini/*.png'))],
    'Kling IMAGE 3.0': [Image.open(f) for f in sorted(glob.glob('colab_upload/kling/*.png'))],
}
res = {}
for name, imgs in groups.items():
    Z = [invert(im) for im in imgs]
    res[name] = (np.array([ring_score(z) for z in Z]), np.array([gauss_score(z) for z in Z]))
    print(f'{name:34s} n={len(imgs):3d}  ring score median {np.median(res[name][0]):.3f}  gauss distance median {np.median(res[name][1]):.4f}')

In [ ]:
base_r, base_g = res['SD clean (no mark)']
print('Lower ring score = more ring-like. p-values: Mann-Whitney vs SD clean (one-sided).')
for name, (r, g) in res.items():
    if name.startswith('SD clean'): continue
    pr = mannwhitneyu(r, base_r, alternative='less').pvalue
    pg = mannwhitneyu(g, base_g, alternative='greater').pvalue
    print(f'{name:34s} ring p={pr:.4f}   gauss p={pg:.4f}')
print()
print('Read it like this:')
print('1. POSITIVE CONTROL must show ring p < 0.01. If not, the test cannot see Tree-Ring marks through this model: stop here.')
print('2. If the control passes and Kling ALSO shows p < 0.01 (and Gemini does not), that is a candidate latent watermark signal.')
print('3. If the control passes and Kling looks like SD clean, no Tree-Ring-type mark is visible through a public model.')